In [2]:
#!/usr/bin/env python

import os
import pandas as pd
import numpy as np

RUN_TAG  = "251110"
OUT_ROOT = f"out/{RUN_TAG}"

# === 1. Point to ONE experiment CSV ===
csv_path = os.path.join(
    OUT_ROOT,
    "results",
    "FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_identical.csv",
)

print(f"Loading: {csv_path}")
df = pd.read_csv(csv_path)

# === 2. Get case START times (for warm-up trimming) ===
df_start = df[df["status"] == "START"][["case_id", "timestamp"]].copy()
df_start = df_start.rename(columns={"timestamp": "start_time"})

# === 3. Completed END events ===
df_complete = df[(df["status"] == "COMPLETE") & (df["activity"] == "END")].copy()
if df_complete.empty:
    raise ValueError("No COMPLETE END events found – check the log or filters.")

# Attach start_time to each completed case
df_complete = df_complete.merge(df_start, on="case_id", how="left")

# === 4. Warm-up trimming (e.g. first 10% of simulated time) ===
t_max = df["timestamp"].max()
warmup_threshold = 0.17 * t_max      # 10% of horizon

df_steady = df_complete[df_complete["start_time"] >= warmup_threshold].copy()

if df_steady.empty:
    raise ValueError("No cases left after warm-up trimming – threshold too strict?")

print(f"Total completed cases: {len(df_complete)}")
print(f"Cases after warm-up trimming: {len(df_steady)}")
print(f"Warm-up threshold (time): {warmup_threshold:.2f}")

# === 5. Var_total on trimmed cases ===
cycle_times = df_steady["cycle_time"].to_numpy(dtype=float)

mu = cycle_times.mean()
var_total = ((cycle_times - mu) ** 2).mean()   # population variance

print(f"\nMean cycle time (steady): {mu:.4f}")
print(f"Var_total (steady, population): {var_total:.4f}")

print("\nQuick summary of cycle times (steady):")
print(df_steady["cycle_time"].describe())


Loading: out/251110\results\FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_identical.csv
Total completed cases: 8425
Cases after warm-up trimming: 6979
Warm-up threshold (time): 5099.99

Mean cycle time (steady): 69.4456
Var_total (steady, population): 1068.8277

Quick summary of cycle times (steady):
count    6979.000000
mean       69.445600
std        32.695274
min         6.710983
25%        45.869595
50%        64.367474
75%        88.043881
max       282.312239
Name: cycle_time, dtype: float64
